# 🧬 Quick Start: Differentiable HS-AFM Simulation

Welcome to **synth-afm**! This tutorial will show you how to generate synthetic High-Speed Atomic Force Microscopy (HS-AFM) data from protein structures using JAX.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/elkins/synth-afm/blob/master/examples/quickstart_afm.ipynb)

## 1. Installation
We'll start by installing `synth-afm` and its dependencies.

In [ ]:
!pip install synth-afm matplotlib

## 2. Setup and Simulation
We'll generate a simple synthetic protein and perform a scan.

In [ ]:
import jax.numpy as jnp
import matplotlib.pyplot as plt
from synth_afm.simulator import AFMSimulator

# 1. Create coordinates (10nm alpha helix)
n_atoms = 100
t = jnp.linspace(0, 10.0, n_atoms)
coords = jnp.stack([5.0 + 2.0 * jnp.cos(t), 5.0 + 2.0 * jnp.sin(t), t], axis=1)

# 2. Initialize Simulator
sim = AFMSimulator(pixel_size=0.5, grid_size=(64, 64), tip_radius=20.0)

# 3. Static Scan (Tip Dilation vs Point Tip)
img_point = sim.scan(coords, use_tip_dilation=False)
img_dilated = sim.scan(coords, use_tip_dilation=True)

# 4. Plot results
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
ax1.imshow(img_point, cmap='viridis')
ax1.set_title("Point Tip (High Res)")
ax2.imshow(img_dilated, cmap='viridis')
ax2.set_title("Dilated Tip (Realistic)")
plt.show()

## 3. Simulating Scanning Lag
Watch how temporal distortion affects a moving molecule.

In [ ]:
# Create a rotating trajectory
trajectory = []
for angle in jnp.linspace(0, jnp.pi/2, 5):
    rot = jnp.array([[jnp.cos(angle), -jnp.sin(angle), 0], [jnp.sin(angle), jnp.cos(angle), 0], [0, 0, 1]])
    trajectory.append(coords @ rot.T)
trajectory = jnp.stack(trajectory)

# Scan movie with realistic lag
movie = sim.scan_movie(trajectory, frames_per_second=2.0)

plt.figure(figsize=(15, 3))
for i in range(5):
    plt.subplot(1, 5, i+1)
    plt.imshow(movie[i])
    plt.title(f"Frame {i}")
    plt.axis('off')
plt.show()